In [1]:
# Import libraries
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
from PIL import Image
import networkx as nx

In [3]:
# Specify the directory path
input_directory = '.../Segmentation watershed/' # Specify the segmentation directory

# Folder to save the results
output_directory = input_directory + 'Centrality map/'
os.makedirs(output_directory, exist_ok=True)

# Function to map centrality back to the image shape for visualization
def map_centrality_to_image(centrality_dict, shape):
    centrality_map = np.zeros(shape)
    for (i, j), centrality_value in centrality_dict.items():
        centrality_map[i, j] = centrality_value
    return centrality_map

# Initialize empty list to hold data for the dataframe
distribution_data = []

# Loop through all files in the directory
for filename in os.listdir(input_directory):
    # Check if the file is a TIFF image
    if filename.lower().endswith('.tif'):
        file_path = os.path.join(input_directory, filename)

        # Step 1: Load binary image
        binary_image = np.array(Image.open(file_path))
        binary_image = binary_image // 255  # Convert pixel values to 0 and 1 (0 for background, 1 for foreground)
        rows, cols = binary_image.shape

        # Step 2: Build graph for foreground pixels (value 1 in binary_image), using 8-connectivity
        G = nx.Graph()

        # Loop over each pixel in the binary image
        for i in range(rows):
            for j in range(cols):
                if binary_image[i, j] == 1:  # Foreground pixel
                    # Add edges for 8-connectivity (up, down, left, right, and diagonals)
                    if i > 0 and binary_image[i-1, j] == 1:  # Up
                        G.add_edge((i, j), (i-1, j))
                    if i < rows - 1 and binary_image[i+1, j] == 1:  # Down
                        G.add_edge((i, j), (i+1, j))
                    if j > 0 and binary_image[i, j-1] == 1:  # Left
                        G.add_edge((i, j), (i, j-1))
                    if j < cols - 1 and binary_image[i, j+1] == 1:  # Right
                        G.add_edge((i, j), (i, j+1))
                    # Diagonal neighbors (Up-Left, Up-Right, Down-Left, Down-Right)
                    if i > 0 and j > 0 and binary_image[i-1, j-1] == 1:  # Up-Left
                        G.add_edge((i, j), (i-1, j-1))
                    if i > 0 and j < cols - 1 and binary_image[i-1, j+1] == 1:  # Up-Right
                        G.add_edge((i, j), (i-1, j+1))
                    if i < rows - 1 and j > 0 and binary_image[i+1, j-1] == 1:  # Down-Left
                        G.add_edge((i, j), (i+1, j-1))
                    if i < rows - 1 and j < cols - 1 and binary_image[i+1, j+1] == 1:  # Down-Right
                        G.add_edge((i, j), (i+1, j+1))

        # Step 3: Compute Degree centrality
        degree_centrality = nx.degree_centrality(G)

        # Step 4: Map centrality back to the 2D image
        degree_centrality_map = map_centrality_to_image(degree_centrality, binary_image.shape)

        # Split the filename by underscores to extract components
        filename_parts = filename.split('_')

        # Extract information based on position
        experiment = filename_parts[0]
        bioreactor = filename_parts[2]

        # Time [h]
        time_str = filename_parts[-7]
        time_h = ''.join(filter(str.isdigit, time_str))

        # Wellplate
        wellplate = [part for part in filename_parts if part.startswith('XY')][0]

        # Save the centrality map as a TIFF file
        output_image_path = os.path.join(output_directory, f'Centrality_{filename}')
        Image.fromarray(degree_centrality_map).save(output_image_path)
        print(f"Successfully saved: {output_image_path}")

        # ============================
        # Distribution logic
        # ============================

        # Identify foreground pixels (assuming non-zero pixels are foreground)
        foreground_pixels = degree_centrality_map[degree_centrality_map > 0]
        num_foreground_pixels = len(foreground_pixels)

        # Multiply all pixel values by the number of foreground pixels
        processed_pixels = (foreground_pixels.astype(np.float32) * num_foreground_pixels) / 8.0

        # Sample 10000 pixels
        sampled_pixels = np.random.choice(processed_pixels, size=10000, replace=False)

        # Store the foreground pixel values and metadata
        for value in sampled_pixels:
            distribution_data.append({
                'Degree Centrality': value,
                'Experiment': experiment,
                'Bioreactor': bioreactor,
                'Time [h]': time_h,
                'Wellplate': wellplate
            })

# Create DataFrame
df_distribution = pd.DataFrame(distribution_data)

# Save the dataframe to CSV in the output directory
output_distribution_path = os.path.join(output_directory, 'Degree_centrality.csv')

df_distribution.to_csv(output_distribution_path, index=False)

print(f"Saved distribution dataframe: {output_distribution_path}")

Successfully saved: H:/Sophie/_DAS/Dataset/test/Centrality map/Centrality_AP00424_AnPY_K7 0124 AN_Pr5_48h_2x_3µmZ_XY04_CFW.tif
Saved distribution dataframe: H:/Sophie/_DAS/Dataset/test/Centrality map/Degree_centrality_distribution.csv
